# 🔬 Cervical Cancer Stage Classification — Training Notebook
### EfficientNet-B3 Hybrid v11 · SIPaKMeD + Herlev · 4-Class Ordinal

**Model:** EfficientNet-B3 + Medical Feature MLP (31-dim) + SE-Attention Fusion  
**Classes:** `Normal → CIN1 → HighGrade → Cancer` (severity-ordered)  
**Loss:** 0.7 × FocalLoss + 0.3 × OrdinalLoss  
**Augmentation:** Adjacent-grade MixUp + CutMix · RandomErasing · ColorJitter  
**Tricks:** SWA · TTA · Progressive backbone unfreeze · WeightedRandomSampler

---
> ⚙️ **Runtime:** Set to *GPU* (Runtime → Change runtime type → GPU) before running.


## Step 0 — Environment & GPU Check

In [ ]:
# ── Install / verify dependencies ────────────────────────────────────────────
import subprocess, sys

PKGS = [
    "torch", "torchvision", "scikit-learn", "scikit-image",
    "opencv-python", "tqdm", "pillow", "numpy", "scipy",
]

def pkg_installed(name):
    try:
        __import__(name.replace("-", "_"))
        return True
    except ImportError:
        return False

missing = [p for p in PKGS if not pkg_installed(p.split("[")[0])]
if missing:
    print(f"Installing: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("✅ All packages already installed.")

# ── GPU diagnostics ───────────────────────────────────────────────────────────
import torch

print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU             : {props.name}")
    print(f"VRAM            : {props.total_memory / 1024**3:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"cuDNN version   : {torch.backends.cudnn.version()}")
else:
    print("⚠️  No GPU detected — training will be very slow.")
    print("   Go to Runtime → Change runtime type → GPU")

print("\n✅ Environment check complete!")


## Step 1 — Clone Repository

In [ ]:
import sys, subprocess, shutil
from pathlib import Path

# ── Detect environment ────────────────────────────────────────────────────────
IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB  = Path("/content").exists() and not IS_KAGGLE
WORK_BASE = Path("/kaggle/working") if IS_KAGGLE else Path("/content")

REPO_URL = "https://github.com/Akshit0707/Cerivcal-Cancer-Stage-Classification"
REPO_DIR = WORK_BASE / "repo"

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")
print(f"Working dir : {WORK_BASE}")

# ── Clone / update ────────────────────────────────────────────────────────────
if REPO_DIR.exists():
    print(f"Repo exists at {REPO_DIR} — pulling latest...")
    r = subprocess.run(["git", "-C", str(REPO_DIR), "pull"], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print(f"Cloning {REPO_URL} ...")
    r = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f"Clone failed:\n{r.stderr}")
    print(r.stdout.strip())

print("\nRepo contents:")
for p in sorted(REPO_DIR.iterdir()):
    marker = "/" if p.is_dir() else ""
    print(f"  {p.name}{marker}")

print(f"\n✅ Repo ready at: {REPO_DIR}")


## Step 2 — Locate `train_hybrid.py` & Configure `sys.path`

In [ ]:
# ── Find train_hybrid.py ─────────────────────────────────────────────────────
hits = list(REPO_DIR.rglob("train_hybrid.py"))
if not hits:
    raise FileNotFoundError(
        "train_hybrid.py not found. "
        "Expected at: <repo>/backend/scripts/train_hybrid.py"
    )

SCRIPT_PATH = hits[0]
SCRIPT_DIR  = SCRIPT_PATH.parent

# ── Infer repo root (layout: <root>/backend/scripts/train_hybrid.py) ─────────
if (SCRIPT_PATH.parents[2] / "backend").exists():
    REPO_ROOT = SCRIPT_PATH.parents[2]
elif SCRIPT_PATH.parents[1].name == "backend":
    REPO_ROOT = SCRIPT_PATH.parents[2]
else:
    REPO_ROOT = SCRIPT_PATH.parent

BACKEND_DIR = REPO_ROOT / "backend"

# ── Inject paths ──────────────────────────────────────────────────────────────
for p in [str(REPO_ROOT), str(BACKEND_DIR), str(SCRIPT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Smoke-test feature_extractor import ──────────────────────────────────────
try:
    from backend.feature_extractor import extract_medical_features
    print("✅ import backend.feature_extractor  — OK")
except ImportError:
    try:
        from feature_extractor import extract_medical_features
        print("✅ import feature_extractor  — OK (fallback)")
    except ImportError as e:
        raise ImportError(
            f"Cannot import feature_extractor: {e}\n"
            "Ensure backend/feature_extractor.py exists in the repo."
        )

print(f"\nScript path : {SCRIPT_PATH}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Backend dir : {BACKEND_DIR}")
print(f"sys.path[0] : {sys.path[0]}")
print("\n✅ Path setup complete!")


## Step 3 — Dataset Setup

The model expects one of two layouts:

```
data/
├── train/
│   ├── Normal/        # SIPaKMeD: Superficial-Intermediate + Parabasal + Metaplastic
│   ├── CIN1/          # SIPaKMeD: Koilocytotic
│   ├── HighGrade/     # SIPaKMeD: Dyskeratotic  (CIN2 + CIN3 merged)
│   └── Cancer/        # Herlev: carcinoma_in_situ
└── val/
    ├── Normal/
    ├── CIN1/
    ├── HighGrade/
    └── Cancer/
```

Or a **flat layout** `data/<ClassName>/*.jpg` (auto-split 80/20).  
Synthetic images should have `syn_` in their filename to enable the is_synthetic feature flag.


In [ ]:
import os
from pathlib import Path
import shutil

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG: set DATA_DIR to your dataset location
# ─────────────────────────────────────────────────────────────────────────────
# Option A — Kaggle: add dataset via "Add Data" panel, path auto-detected below
# Option B — Colab: upload to /content/data or mount Google Drive
# Option C — Manual: set DATA_DIR_OVERRIDE directly
DATA_DIR_OVERRIDE = None   # e.g. "/content/drive/MyDrive/sipakmed_4class"

LOCAL_DATA = WORK_BASE / "data"

SEVERITY_ORDER = ["Normal", "CIN1", "HighGrade", "Cancer"]

def find_dataset_root():
    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path("/content/drive/MyDrive"),
        Path("/content"),
        REPO_DIR / "data",
        Path("."),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        # Prefer pre-split layout
        for train_dir in root.rglob("train"):
            if train_dir.is_dir() and (train_dir.parent / "val").is_dir():
                return train_dir.parent
        # Fall back to flat layout matching expected class names
        for d in root.rglob("Normal"):
            if d.is_dir():
                parent = d.parent
                if any((parent / c).is_dir() for c in ["CIN1", "HighGrade", "Cancer"]):
                    return parent
    return None

# ── Resolve DATA_DIR ──────────────────────────────────────────────────────────
if DATA_DIR_OVERRIDE:
    DATA_DIR = Path(DATA_DIR_OVERRIDE)
    print(f"📂 Using override: {DATA_DIR}")
elif LOCAL_DATA.exists() and (LOCAL_DATA / "train").exists():
    DATA_DIR = LOCAL_DATA
    print(f"✅ Data already at: {DATA_DIR}")
else:
    found = find_dataset_root()
    if found is None:
        raise FileNotFoundError(
            "❌ Dataset not found.\n"
            "• Kaggle: click 'Add Data' and attach your cervical cancer dataset.\n"
            "• Colab : upload your data or set DATA_DIR_OVERRIDE above."
        )
    print(f"Found dataset at: {found}")
    if not (LOCAL_DATA.exists() and LOCAL_DATA.samefile(found) if LOCAL_DATA.exists() else False):
        print("📦 Copying to working directory...")
        if LOCAL_DATA.exists():
            shutil.rmtree(LOCAL_DATA)
        shutil.copytree(str(found), str(LOCAL_DATA))
        print("✅ Copied.")
    DATA_DIR = LOCAL_DATA

# ── Verify & summarise ────────────────────────────────────────────────────────
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def count_split(split_dir: Path):
    result = {}
    if not split_dir.exists():
        return result
    for cls_dir in sorted(split_dir.iterdir()):
        if cls_dir.is_dir():
            n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() in IMG_EXTS)
            syn = sum(1 for f in cls_dir.iterdir()
                      if f.suffix.lower() in IMG_EXTS and "syn_" in f.name)
            result[cls_dir.name] = {"total": n, "synthetic": syn}
    return result

print(f"\n📊 Dataset: {DATA_DIR}")
grand_total = 0
for split in ["train", "val"]:
    counts = count_split(DATA_DIR / split)
    if counts:
        print(f"\n  {split.upper()}/")
        print(f"  {'Class':<20} {'Total':>7} {'Synthetic':>10}")
        print(f"  {'-'*40}")
        for cls in SEVERITY_ORDER:
            if cls in counts:
                c = counts[cls]
                print(f"  {cls:<20} {c['total']:>7}  {c['synthetic']:>9}")
                grand_total += c["total"]
        others = [c for c in counts if c not in SEVERITY_ORDER]
        for cls in others:
            c = counts[cls]
            print(f"  {cls:<20} {c['total']:>7}  {c['synthetic']:>9}  ⚠️ unknown class")
            grand_total += c["total"]

print(f"\n  Grand total: {grand_total} images")
print(f"\n✅ Dataset ready at: {DATA_DIR}")


## Step 4 — Training Configuration

Adjust hyperparameters below before launching training.  
Defaults are tuned for a T4/V100 GPU with the full SIPaKMeD + Herlev dataset.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Hyperparameters — tune for your GPU / dataset size
# ─────────────────────────────────────────────────────────────────────────────

EPOCHS         = 100   # Early stopping kicks in around epoch 40–70
BATCH_SIZE     = 32    # 32 → T4 (16 GB) | 64 → A100 (40 GB) | 16 → smaller GPU
LEARNING_RATE  = 2e-4  # Backbone gets 1/10th (2e-5) automatically
PATIENCE       = 20    # Early-stopping patience (epochs without improvement)
NUM_WORKERS    = 4     # DataLoader workers; reduce to 2 if you see stall warnings
SEED           = 42

# ── Optional toggles (mirror train_hybrid.py CLI flags) ──────────────────────
USE_TTA    = True    # Test-Time Augmentation  (5 flips/rotations every 5 epochs)
USE_SWA    = True    # Stochastic Weight Averaging  (starts at epoch 45)
USE_CUTMIX = True    # CutMix  (blended with adjacent-grade MixUp from epoch 20)
USE_SAM    = False   # Sharpness-Aware Minimisation  (slower, rarely needed)

# ── Output directories ────────────────────────────────────────────────────────
CHECKPOINT_DIR = WORK_BASE / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ── Print summary ─────────────────────────────────────────────────────────────
import torch

print("=" * 60)
print("TRAINING CONFIGURATION  (v11 · 4-class ordinal)")
print("=" * 60)
print(f"  Data dir        : {DATA_DIR}")
print(f"  Checkpoint dir  : {CHECKPOINT_DIR}")
print(f"  Epochs          : {EPOCHS}  (patience={PATIENCE})")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Learning rate   : {LEARNING_RATE}  (backbone × 0.1)")
print(f"  Workers         : {NUM_WORKERS}")
print(f"  TTA             : {USE_TTA}")
print(f"  SWA             : {USE_SWA}  (starts epoch 45)")
print(f"  CutMix          : {USE_CUTMIX}")
print(f"  SAM             : {USE_SAM}")
print(f"  Seed            : {SEED}")
print(f"  GPU             : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ⚠️'}")
print(f"  AMP             : {'Enabled' if torch.cuda.is_available() else 'Disabled'}")
print("=" * 60)
print("\n✅ Configuration ready — run Step 5 to start training.")


## Step 5 — Launch Training

Runs `train_hybrid.py` as a subprocess so its output streams live to the cell.  
Training typically completes in **2–4 hours on a T4 GPU** (100 epochs, early stop).


In [ ]:
import os, subprocess, sys

# ── Build PYTHONPATH ──────────────────────────────────────────────────────────
extra = [str(REPO_ROOT), str(BACKEND_DIR), str(SCRIPT_DIR)]
env   = os.environ.copy()
pp    = env.get("PYTHONPATH", "")
env["PYTHONPATH"]     = ":".join(extra) + (":" + pp if pp else "")
env["PYTHONUNBUFFERED"] = "1"   # live output

# ── Build CLI command ─────────────────────────────────────────────────────────
cmd = [
    sys.executable, str(SCRIPT_PATH),
    "--data-dir",                str(DATA_DIR),
    "--checkpoint-dir",          str(CHECKPOINT_DIR),
    "--epochs",                  str(EPOCHS),
    "--batch-size",              str(BATCH_SIZE),
    "--learning-rate",           str(LEARNING_RATE),
    "--early-stopping-patience", str(PATIENCE),
    "--num-workers",             str(NUM_WORKERS),
    "--seed",                    str(SEED),
]
if not USE_TTA:    cmd.append("--no-tta")
if not USE_SWA:    cmd.append("--no-swa")
if USE_SAM:        cmd.append("--use-sam")
if not USE_CUTMIX: cmd.append("--no-cutmix")

print("Command:")
print("  " + " ".join(cmd))
print("\n" + "=" * 70)
print("TRAINING LOG")
print("=" * 70 + "\n")

# ── Run — streams stdout/stderr live ─────────────────────────────────────────
proc = subprocess.Popen(
    cmd,
    cwd=str(REPO_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()

if proc.returncode == 0:
    print("\n✅ Training completed successfully!")
else:
    raise RuntimeError(
        f"Training script exited with code {proc.returncode}.\n"
        "Scroll up to find the error, or check CHECKPOINT_DIR for partial checkpoints."
    )


## Step 6 — Inspect Saved Checkpoints

In [ ]:
import torch
from pathlib import Path

print(f"📁 Checkpoint directory: {CHECKPOINT_DIR}\n")

# ── List all .pt files ────────────────────────────────────────────────────────
pt_files = sorted(CHECKPOINT_DIR.rglob("*.pt"))
if not pt_files:
    print("⚠️  No .pt checkpoints found. Check that training completed.")
else:
    print(f"{'File':<30} {'Size (MB)':>10}")
    print("-" * 42)
    for p in pt_files:
        size = p.stat().st_size / 1e6
        print(f"  {p.name:<28} {size:>8.1f} MB")

# ── Load best_model.pt and print metadata ─────────────────────────────────────
best_ckpt_path = CHECKPOINT_DIR / "best_model.pt"
swa_ckpt_path  = CHECKPOINT_DIR / "swa_model.pt"

def show_ckpt(path, label):
    if not path.exists():
        print(f"\n  {label}: not found")
        return None
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print(f"\n{'='*50}")
    print(f"  {label}  ({path.name})")
    print(f"{'='*50}")
    print(f"  Version      : {ckpt.get('version', 'N/A')}")
    print(f"  Epoch        : {ckpt.get('epoch', 'N/A')}")
    print(f"  Classes      : {ckpt.get('class_names', 'N/A')}")
    print(f"  Num classes  : {ckpt.get('num_classes', 'N/A')}")
    print(f"  Num features : {ckpt.get('num_features', 'N/A')}")
    print(f"  Input size   : {ckpt.get('input_size', 'N/A')}")
    print(f"  Val accuracy : {ckpt.get('val_acc', 0):.2f}%")
    print(f"  Balanced acc : {ckpt.get('val_bacc', 0):.2f}%")
    print(f"  Macro F1     : {ckpt.get('macro_f1', 0):.4f}")
    print(f"  SWA          : {ckpt.get('use_swa', False)}")
    print(f"  Severity     : {ckpt.get('severity_order', 'N/A')}")
    return ckpt

best_ckpt = show_ckpt(best_ckpt_path, "BEST MODEL")
swa_ckpt  = show_ckpt(swa_ckpt_path,  "SWA MODEL")


## Step 7 — Plot Training History

In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

hist_path = CHECKPOINT_DIR / "history.json"
if not hist_path.exists():
    print("⚠️  history.json not found. Run training first.")
else:
    with open(hist_path) as f:
        hist = json.load(f)

    epochs_ran = len(hist["tr_loss"])
    xs = list(range(1, epochs_ran + 1))

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle("Training History — EfficientNet-B3 Hybrid v11", fontsize=14, fontweight="bold")
    gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

    # Loss
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(xs, hist["tr_loss"], label="Train loss", color="#2196F3")
    ax.plot(xs, hist["va_loss"], label="Val loss",   color="#F44336")
    ax.set_title("Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(alpha=0.3)

    # Accuracy
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(xs, hist["tr_acc"], label="Train acc",      color="#2196F3")
    ax.plot(xs, hist["va_acc"], label="Val acc",        color="#F44336")
    ax.plot(xs, hist["va_bacc"], label="Val bal. acc",  color="#FF9800", linestyle="--")
    ax.set_title("Accuracy"); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
    ax.legend(); ax.grid(alpha=0.3)

    # Macro F1
    ax = fig.add_subplot(gs[1, 0])
    ax.plot(xs, hist["va_f1"], color="#4CAF50", linewidth=2)
    best_ep = int(np.argmax(hist["va_f1"])) + 1
    best_f1 = max(hist["va_f1"])
    ax.axvline(best_ep, color="gray", linestyle=":", label=f"Best ep={best_ep} F1={best_f1:.4f}")
    ax.set_title("Validation Macro F1"); ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
    ax.legend(); ax.grid(alpha=0.3)

    # Balanced accuracy
    ax = fig.add_subplot(gs[1, 1])
    ax.plot(xs, hist["va_bacc"], color="#9C27B0", linewidth=2)
    best_ba_ep = int(np.argmax(hist["va_bacc"])) + 1
    best_ba    = max(hist["va_bacc"])
    ax.axvline(best_ba_ep, color="gray", linestyle=":",
               label=f"Best ep={best_ba_ep} bAcc={best_ba:.2f}%")
    ax.set_title("Validation Balanced Accuracy"); ax.set_xlabel("Epoch"); ax.set_ylabel("bAcc (%)")
    ax.legend(); ax.grid(alpha=0.3)

    plt.savefig(str(CHECKPOINT_DIR / "training_history.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\n  Best epoch     : {best_ep}  (Macro F1 = {best_f1:.4f})")
    print(f"  Best bal. acc  : {best_ba:.2f}% at epoch {best_ba_ep}")
    print(f"  Plot saved to  : {CHECKPOINT_DIR / 'training_history.png'}")


## Step 8 — Inference on a Sample Image

Loads the best checkpoint and runs a forward pass on one image from the validation set.  
You can point `INFERENCE_IMAGE` at any path to test on your own images.


In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import numpy as np
import importlib.util

# ── Load model class directly from script (avoids import issues) ──────────────
spec = importlib.util.spec_from_file_location("train_hybrid", str(SCRIPT_PATH))
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
EfficientNetHybrid = mod.EfficientNetHybrid
NUM_FEATURES       = mod.NUM_FEATURES   # 31
INPUT_SIZE         = mod.INPUT_SIZE     # 300

# ── Reload feature extractor ──────────────────────────────────────────────────
try:
    from backend.feature_extractor import extract_medical_features
except ImportError:
    from feature_extractor import extract_medical_features

# ── Load checkpoint ───────────────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = CHECKPOINT_DIR / "best_model.pt"

if not ckpt_path.exists():
    print("⚠️  best_model.pt not found. Run Step 5 (training) first.")
else:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    class_names   = ckpt["class_names"]        # e.g. ['Normal','CIN1','HighGrade','Cancer']
    num_classes   = ckpt["num_classes"]
    num_feats     = ckpt.get("num_features", NUM_FEATURES)
    severity_order = ckpt.get("severity_order", class_names)

    model = EfficientNetHybrid(num_classes=num_classes, num_features=num_feats)

    # Handle SWA state dict key prefix
    sd = ckpt["model_state_dict"]
    if any(k.startswith("module.") for k in sd):
        sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=False)
    model.to(device).eval()
    print(f"✅ Loaded: {ckpt_path.name}  (epoch {ckpt.get('epoch','?')})")
    print(f"   Classes : {class_names}")
    print(f"   Val F1  : {ckpt.get('macro_f1', 0):.4f}  |  "
          f"bAcc: {ckpt.get('val_bacc', 0):.2f}%  |  Acc: {ckpt.get('val_acc', 0):.2f}%")

    # ── Pick a sample from val set (or set INFERENCE_IMAGE manually) ──────────
    INFERENCE_IMAGE = None   # e.g. "/content/data/val/CIN1/img001.png"

    if INFERENCE_IMAGE is None:
        for cls_dir in sorted((DATA_DIR / "val").iterdir()):
            if cls_dir.is_dir():
                imgs = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
                if imgs:
                    INFERENCE_IMAGE = str(imgs[0])
                    true_label = cls_dir.name
                    break
    else:
        from pathlib import Path as _P
        true_label = _P(INFERENCE_IMAGE).parent.name

    if INFERENCE_IMAGE is None:
        print("No images found in val set.")
    else:
        img = Image.open(INFERENCE_IMAGE).convert("RGB")

        # Preprocess
        tf = transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        img_t  = tf(img).unsqueeze(0).to(device)

        # 30 medical features + 1 synthetic flag
        med_feats = extract_medical_features(img)
        is_syn    = 1.0 if "syn_" in str(INFERENCE_IMAGE) else 0.0
        feats     = np.append(np.array(med_feats, dtype=np.float32), is_syn)
        feat_t    = torch.FloatTensor(feats).unsqueeze(0).to(device)

        with torch.no_grad():
            logits, ord_logits = model(img_t, feat_t)
            probs = F.softmax(logits, dim=-1)[0].cpu().numpy()

        pred_idx = int(probs.argmax())
        pred_cls = class_names[pred_idx]

        print(f"\n{'─'*50}")
        print(f"  Image          : {INFERENCE_IMAGE}")
        print(f"  True label     : {true_label}")
        print(f"  Predicted      : {pred_cls}  (conf: {probs[pred_idx]*100:.1f}%)")
        print(f"\n  Class probabilities:")
        for i, (c, p) in enumerate(zip(class_names, probs)):
            bar = "█" * int(p * 30)
            marker = " ← predicted" if i == pred_idx else ""
            print(f"    {c:<12} {p*100:5.1f}%  {bar}{marker}")

        # Ordinal thresholds — P(Y >= k) for k=1,2,3
        ord_probs = torch.sigmoid(ord_logits)[0].cpu().numpy()
        print(f"\n  Ordinal thresholds (P(severity ≥ k)):")
        for k, (cls_name, p) in enumerate(zip(class_names[1:], ord_probs), 1):
            print(f"    P(≥ {cls_name:<10}) = {p:.3f}")
        print(f"{'─'*50}")


## Step 9 — Package & Download Checkpoints

In [ ]:
import zipfile, shutil
from pathlib import Path

zip_path = WORK_BASE / "trained_model_v11.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(CHECKPOINT_DIR.rglob("*")):
        if p.is_file():
            arcname = p.relative_to(CHECKPOINT_DIR)
            zf.write(p, arcname=arcname)

size_mb = zip_path.stat().st_size / 1e6
print(f"✅ Zip created: {zip_path}  ({size_mb:.1f} MB)")
print(f"   Contents:")
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f"     {name:<35}  {info.file_size/1e6:.1f} MB")

# ── Auto-download ─────────────────────────────────────────────────────────────
try:
    from google.colab import files
    files.download(str(zip_path))
    print("\n✅ Download started (Colab).")
except ImportError:
    print(f"\nℹ️  Kaggle: open the Output panel on the right → download trained_model_v11.zip")
    print(f"   Path: {zip_path}")
